# 04b. 실제 중계 스크립트 → 말투 예시 / 표현 은행 후보

**로컬 커널에서 실행** (GPU 불필요, 표준 라이브러리만 사용)

- 입력: 유튜브 자동 자막을 복사한 중계 스크립트 (중계사 저작물 → 프로젝트 폴더 밖, 비공개)
- 출력도 원문 조각을 포함하므로 같은 비공개 폴더의 `derived/`에 저장

1. 타임스탬프 제거, 문장 분리
2. 문장별 판정 키워드 태깅 → 경기별 빈도
3. 실제 카운트 표현 모으기 ("투앤원" 등)
4. 투구 중계가 몰린 구간 → LLM 퓨샷용 말투 예시
5. 판정별 짧은 문장 → 템플릿 표현 은행 후보

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

# 원본 스크립트는 저작물이므로 kbo-listen 밖에 둔다
SRC = Path(r"C:\Users\jhm99\OneDrive\Desktop\colab 과제\kbo")
OUT = SRC / "derived"
OUT.mkdir(exist_ok=True)

# "13:4213분 42초", "2:002분", "0:055초" 같은 줄 앞 타임스탬프
TS = re.compile(r"^\d{1,2}(?::\d{2}){1,2}(?:\d+시간\s?)?(?:\d+분\s?)?(?:\d+초)?")


def load_sentences(path):
    # 자막 줄은 문장 중간에서 끊기므로 전부 이어 붙인 뒤 문장부호로 다시 자른다
    lines = [TS.sub("", line).strip() for line in path.read_text(encoding="utf-8").splitlines()]
    text = " ".join(line for line in lines if line)
    return [s.strip() for s in re.split(r"(?<=[.?!])\s+", text) if s.strip()]


games = {p.stem: load_sentences(p) for p in sorted(SRC.glob("*.txt"))}
for name, sents in games.items():
    print(name, len(sents), "문장")

print()
for s in games[next(iter(games))][200:210]:
    print("-", s)

In [ ]:
# 대략적인 키워드 규칙. 자동 자막 오류(3진, 수윙 등)도 일부 포함
TAGS = {
    # 땅볼·뜬볼의 '볼이고요'는 제외, '빠졌'은 판정 어미가 붙은 경우만 (예: '60경기 빠졌으니까요' 제외)
    "ball":      r"(?<![땅뜬])볼(?:입니다|이고요|이에요|이죠)|빠집니다|빠졌(?:고요|습니다|네요)|빠지면서|골라냅|골라냈",
    "strike":    r"스트라이크(?!\s?존)",
    "swing":     r"헛스윙|헛 스윙|수윙",
    "foul":      r"파울|커트",
    "strikeout": r"삼진|3진",
    "walk":      r"볼넷|포볼",
    "hbp":       r"몸에 맞|사구|데드볼",
    "hit":       r"안타|홈런|2루타|3루타|넘어갑니다",
    "out_play":  r"잡아서|처리합니다|잡아냅니다|잡았습니다|땅볼|뜬공|플라이",
    "count":     r"풀\s?카운트|볼카운트|(?:노|원|투|쓰리|쓰|[0-3])\s?(?:볼|앤)\s?(?:노|원|투|[0-2])",
    "outs":      r"(?:노|원|투|쓰리)\s?아웃|(?:첫|두|세)\s?번째 아웃",
    "ordinal":   r"초구|\d+구째|(?:두|세|네|다섯|여섯) 번째 공",
}
TAG_RE = {k: re.compile(v) for k, v in TAGS.items()}


def tags_of(s):
    return {k for k, r in TAG_RE.items() if r.search(s)}


# 문장마다 태그를 한 번만 계산해 둔다
tagged = {name: [(s, tags_of(s)) for s in sents] for name, sents in games.items()}

# 경기별 태그가 붙은 문장 수 ("원 스트라이크"처럼 count와 strike에 같이 잡히는 문장도 있음)
print("문장 수 기준")
for name, items in tagged.items():
    c = Counter(t for _, tags in items for t in tags)
    print(f"\n{name}")
    print("  " + ", ".join(f"{k} {c[k]}" for k in TAGS))

In [ ]:
# 실제 캐스터의 카운트 표현 → 템플릿 count_words() 변형 후보
count_forms = Counter(
    m.group(0)
    for items in tagged.values()
    for s, _ in items
    for m in TAG_RE["count"].finditer(s)
)
for form, n in count_forms.most_common(30):
    print(f"{n:>3}  {form}")

In [ ]:
# 투구 중계가 몰린 구간 → LLM 퓨샷 말투 예시
# 실제 판정 멘트는 짧다. 긴 문장 속 키워드(해설위원의 카운트 분석 등)는 점수에서 뺀다
PLAY = set(TAGS)
SHORT = 40


def play_windows(items, size=5, top=15):
    scored = []
    for i in range(len(items) - size + 1):
        score = sum(len(tags & PLAY) for s, tags in items[i:i + size] if len(s) <= SHORT)
        scored.append((score, i))
    scored.sort(reverse=True)

    picked, used = [], set()
    for score, i in scored:
        if len(picked) == top:
            break
        if used & set(range(i, i + size)):  # 겹치는 구간은 건너뜀
            continue
        picked.append({"score": score, "text": " ".join(s for s, _ in items[i:i + size])})
        used.update(range(i, i + size))
    return picked


style = {name: play_windows(items) for name, items in tagged.items()}

for name, wins in style.items():
    print(f"===== {name}")
    for w in wins[:3]:
        print(f"[{w['score']}] {w['text']}\n")

(OUT / "style_examples.json").write_text(json.dumps(style, ensure_ascii=False, indent=1), encoding="utf-8")
print("저장:", OUT / "style_examples.json")

In [ ]:
# 판정별 짧은 문장 모으기 → 04 템플릿(CALL, IN_PLAY 등)에 넣을 표현 후보
FILLER = re.compile(r"^(?:자|네|예|아|어|음|그렇죠|그렇습니다)[,.\s]+")


def strip_filler(s):
    prev = None
    while prev != s:
        prev, s = s, FILLER.sub("", s).strip()
    return s


BANK_TAGS = ["ball", "strike", "swing", "foul", "strikeout", "walk", "hbp", "hit", "out_play", "outs"]
candidates = {}
for tag in BANK_TAGS:
    short = Counter(
        strip_filler(s)
        for items in tagged.values()
        for s, tags in items
        if tag in tags and len(s) <= 45  # 짧은 문장만: 긴 문장은 대부분 해설 잡담
    )
    candidates[tag] = [s for s, _ in short.most_common()]

for tag, sents in candidates.items():
    print(f"===== {tag} ({len(sents)}개)")
    for s in sents[:10]:
        print("  -", s)

(OUT / "phrase_candidates.json").write_text(json.dumps(candidates, ensure_ascii=False, indent=1), encoding="utf-8")
print("\n저장:", OUT / "phrase_candidates.json")